In [1]:
import pandas as pd
import networkx as nx
from itertools import combinations
from collections import defaultdict

In [3]:
cast_clean = pd.read_csv(
    "data/processed/production_cast_clean.csv"
)

print(cast_clean.shape)
cast_clean.head()

(115632, 4)


,performer_id,performer_name,character,production_id
0,A_473388,Dot Campbell,Chorus,10336
1,A_34408,Raymond Campbell,Skinny,10336
2,A_473389,Alice Carter,Chorus,10336
3,A_35813,Louis Cole,Jimmy,10336
4,A_438924,Craddock and Shadney,Specialty,10336


In [4]:
edge_weights = defaultdict(int)

for production_id, group in cast_clean.groupby("production_id"):

    performers = group["performer_id"].unique()

    for a, b in combinations(sorted(performers), 2):
        edge_weights[(a,b)] += 1

In [5]:
G = nx.Graph()

for performer_id, name in (
    cast_clean[
        ["performer_id", "performer_name"]
    ]
    .drop_duplicates()
    .values
):
    G.add_node(
        performer_id,
        performer_name=name
    )


for (a,b), weight in edge_weights.items():

    G.add_edge(
        a,
        b,
        weight=weight
    )

In [6]:
print(G.number_of_nodes())
print(G.number_of_edges())

55261
1565292


In [8]:
nx.write_graphml(
    G,
    "data/processed/broadway_network_clean.graphml"
)

In [10]:
comparison = pd.DataFrame({
    "metric": [
        "nodes",
        "edges",
        "density",
        "components"
    ],
    "raw": [
        55263,
        1566284,
        0.00102575,
        nx.number_connected_components(nx.read_graphml(
            "data/processed/broadway_network.graphml"
        ))
    ],
    "clean": [
        G.number_of_nodes(),
        G.number_of_edges(),
        nx.density(G),
        nx.number_connected_components(G)
    ]
})

In [11]:
comparison

,metric,raw,clean
0,nodes,5.526300e+04,5.526100e+04
1,edges,1.566284e+06,1.565292e+06
2,density,1.025750e-03,1.025170e-03
3,components,2.850000e+02,2.850000e+02


In [14]:
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("Density:", nx.density(G))
print(
    "Connected components:",
    nx.number_connected_components(G)
)

Nodes: 55261
Edges: 1565292
Density: 0.001025169651446366
Connected components: 285


In [15]:
degree_df = (
    pd.DataFrame(
        G.degree(),
        columns=["performer_id", "degree"]
    )
    .sort_values(
        "degree",
        ascending=False
    )
    .head(20)
)

degree_df

,performer_id,degree
198,A_53589,720
401,A_68475,700
3932,A_21639,698
8265,A_35584,692
21693,A_21617,675
4393,A_52148,652
2525,A_67079,649
11659,A_39487,642
36366,A_66920,640
2028,A_14598,634


In [16]:
degree_df.merge(
    cast_clean[
        ["performer_id", "performer_name"]
    ].drop_duplicates(),
    on="performer_id"
)

,performer_id,degree,performer_name
0,A_53589,720,Victor Moore
1,A_68475,700,Le Roi Operti
2,A_21639,698,Dennis King
3,A_35584,692,Dudley Clements
4,A_21617,675,David Wayne
5,A_52148,652,Charles McClelland
6,A_67079,649,Robert Chisholm
7,A_39487,642,Maurice Ellis
8,A_66920,640,Philip Bosco
9,A_14598,634,Clarence Derwent


In [17]:
components = sorted(
    nx.connected_components(G),
    key=len,
    reverse=True
)

print("Largest component:", len(components[0]))
print("Second largest:", len(components[1]))
print("Number of components:", len(components))

largest_cc = G.subgraph(components[0]).copy()

print(
    largest_cc.number_of_nodes(),
    largest_cc.number_of_edges()
)

Largest component: 52874
Second largest: 104
Number of components: 285
52874 1538675
